# Evaluate CUPAC before the outcome is available

[All How-to Guides](README.md)

You have two historical periods and the new experiment outcome is still pending. Use a virtual target to estimate potential variance reduction from historical data.

## 1. Define the virtual target

The dataset does not yet contain `y`. Set `parent="y"` on both historical `PreTargetRole` columns and specify `cofounders` on the most recent lag. Use `FeatureRole(parent=..., lag=...)` to link features to their corresponding periods.

In [1]:
from hypex.dataset import Dataset, ExperimentData, FeatureRole, PreTargetRole
from hypex.ml import CUPACExecutor
from hypex.reporters import CupacReporter
from hypex.utils.tutorial_data_creation import DataGenerator

frame = DataGenerator(n_samples=800, seed=42).generate().rename(
    columns={"y0_lag_1": "y_lag1", "y0_lag_2": "y_lag2"}
)
roles = {
    "y_lag1": PreTargetRole(parent="y", lag=1, cofounders=["X1", "X2"]),
    "y_lag2": PreTargetRole(parent="y", lag=2),
    "X1_lag1": FeatureRole(parent="X1", lag=1),
    "X1_lag2": FeatureRole(parent="X1", lag=2),
    "X2_lag1": FeatureRole(parent="X2", lag=1),
    "X2_lag2": FeatureRole(parent="X2", lag=2),
}
data = Dataset(roles=roles, data=frame[list(roles)])
data.columns

['y_lag1', 'y_lag2', 'X1_lag1', 'X1_lag2', 'X2_lag1', 'X2_lag2']

The dataset contains only historical columns. The lags represent the virtual target `y`. The model predicts its most recent lag from the earlier period.

## 2. Evaluate the model on historical data

Run `CUPACExecutor` directly and read the results with `CupacReporter`. `cupac_models` specifies the candidates for historical cross-validation.

`enable_cupac` is an argument to `ABTest`. A direct call to `CUPACExecutor` runs CUPAC, and `cupac_models` limits the model search.

In [2]:
history = CUPACExecutor(cupac_models=["linear", "ridge"]).execute(
    ExperimentData(data)
)
report = CupacReporter().report(history)
report["variance_reductions"]

,target,best_model,variance_reduction_cv,variance_reduction_real
0,y,linear,65.3467,None


`variance_reduction_cv` estimates variance reduction on historical data. `variance_reduction_real` is empty because the experiment outcome is still unavailable.

Check which historical features the selected model used.

In [3]:
report["feature_importances"]

,target,feature,importance,model
0,y,X1_lag2,-0.031754,linear
1,y,X2_lag2,0.148898,linear
2,y,y_lag2,0.808093,linear


Feature names refer to the training period. For a linear model, the values are coefficients averaged across folds.

Once the new outcome is available, see the [full CUPAC example with ABTest](../quick_tutorials/CUPED_CUPAC.ipynb#cupac).

With three historical periods, add lag 3 for the outcome and each feature so that the lags are consecutive. HypEx uses the transitions from lag 3 to lag 2 and from lag 2 to lag 1.